### Red sin fugas

In [ ]:
import pandas as pd 
import random

In [2]:

# Estructura de la red: Definimos quién es el padre de quién
# Formato: 'Hijo': 'Padre'
red_estructura = {
    'Matriz_A': None,   # La fuente no tiene padre
    'Nodo_B': 'Matriz_A',
    'Nodo_C': 'Matriz_A',
    'Casa_D': 'Nodo_B',
    'Casa_E': 'Nodo_B',
    'Casa_F': 'Nodo_C',
    'Casa_G': 'Nodo_C'
}

# Identificamos cuáles son los nodos finales (hojas) donde se consume agua
nodos_finales = ['Casa_D', 'Casa_E', 'Casa_F', 'Casa_G']

In [4]:
def generar_datos_red_sin_fugas():
    data = []
    
    # 1. Asignar Caudales (Consumo) a los nodos finales
    # Generamos un consumo aleatorio entre 10 y 50 Litros/minuto para cada casa
    caudales = {nodo: 0 for nodo in red_estructura.keys()}
    
    for nodo in nodos_finales:
        caudales[nodo] = round(random.uniform(10.0, 50.0), 2)

    # 2. Calcular Caudal hacia atrás (Balance de Masas)
    # Sumamos los hijos para obtener el caudal de los padres
    # Iteramos varias veces para asegurar que el flujo suba hasta la matriz
    for _ in range(3): 
        for hijo, padre in red_estructura.items():
            if padre:
                # El caudal del padre aumenta con el del hijo (lógica simplificada para árbol)
                # Nota: En un script real recursivo esto se hace en una pasada, 
                # aquí lo hacemos iterativo simple.
                pass 
    
    # Recalculo preciso usando estructura de árbol inversa
    # Reiniciamos nodos no finales
    for nodo in red_estructura:
        if nodo not in nodos_finales:
            caudales[nodo] = 0
            
    # Propagación de abajo hacia arriba (Bottom-up)
    # D y E suman a B; F y G suman a C; B y C suman a A.
    caudales['Nodo_B'] = caudales['Casa_D'] + caudales['Casa_E']
    caudales['Nodo_C'] = caudales['Casa_F'] + caudales['Casa_G']
    caudales['Matriz_A'] = caudales['Nodo_B'] + caudales['Nodo_C']

    # 3. Simular Presiones (La presión cae desde la matriz)
    # Presión inicial en Bar (ej. 5 Bar en la bomba)
    presiones = {'Matriz_A': 5.0}
    
    # Propagación de arriba hacia abajo (Top-down)
    # Cada tramo pierde un poco de presión (fricción ficticia)
    cola_proceso = ['Matriz_A']
    while cola_proceso:
        padre = cola_proceso.pop(0)
        # Buscar hijos de este padre
        hijos = [k for k, v in red_estructura.items() if v == padre]
        
        for hijo in hijos:
            perdida = random.uniform(0.1, 0.3) # Perdida de presión por tubería
            presiones[hijo] = round(presiones[padre] - perdida, 2)
            cola_proceso.append(hijo)

    # 4. Preparar datos para Excel
    for nodo in red_estructura:
        tipo = "Final" if nodo in nodos_finales else "Intermedio"
        if nodo == 'Matriz_A': tipo = "Fuente"
        
        data.append({
            'Nodo_ID': nodo,
            'Tipo': tipo,
            'Conectado_A_Padre': red_estructura[nodo] if red_estructura[nodo] else "N/A",
            'Presion_Sensor_Bar': presiones[nodo],
            'Caudal_Sensor_Lmin': round(caudales[nodo], 2),
            'Estado': 'Normal'
        })
        
    return data

In [6]:
# --- EJECUCIÓN ---
datos_simulados = generar_datos_red_sin_fugas()
df = pd.DataFrame(datos_simulados)

# Ordenar para que la Matriz salga primero
df = df.sort_values(by=['Caudal_Sensor_Lmin'], ascending=False)

print("Vista previa de los datos generados:")
print(df)

# Exportar a Excel
nombre_archivo = 'datos_red_agua_etapa1.xlsx'
df.to_excel(nombre_archivo, index=False)
print(f"\nArchivo exportado exitosamente: {nombre_archivo}")

Vista previa de los datos generados:
    Nodo_ID        Tipo Conectado_A_Padre  Presion_Sensor_Bar  \
0  Matriz_A      Fuente               N/A                5.00   
1    Nodo_B  Intermedio          Matriz_A                4.87   
2    Nodo_C  Intermedio          Matriz_A                4.80   
4    Casa_E       Final            Nodo_B                4.70   
6    Casa_G       Final            Nodo_C                4.50   
5    Casa_F       Final            Nodo_C                4.67   
3    Casa_D       Final            Nodo_B                4.61   

   Caudal_Sensor_Lmin  Estado  
0              119.45  Normal  
1               67.97  Normal  
2               51.48  Normal  
4               48.09  Normal  
6               31.40  Normal  
5               20.08  Normal  
3               19.88  Normal  

Archivo exportado exitosamente: datos_red_agua_etapa1.xlsx


### Red con fugas

In [7]:
# --- FUNCIÓN 1: GENERAR ESCENARIO CON FUGA ---
def generar_red_con_fuga():
    # 1. Generamos consumos normales para las casas
    caudales = {nodo: 0.0 for nodo in red_estructura}
    for nodo in nodos_finales:
        caudales[nodo] = round(random.uniform(10.0, 50.0), 2)
    
    # 2. SELECCIONAR LA FUGA
    # Elegimos un nodo al azar (cualquiera menos la Matriz) para que tenga la fuga
    posibles_fugas = list(red_estructura.keys())
    posibles_fugas.remove('Matriz_A')
    nodo_con_fuga = random.choice(posibles_fugas)
    
    # Magnitud de la fuga (ej. entre 20 y 100 litros extra)
    valor_fuga = round(random.uniform(20.0, 100.0), 2)
    
    # 3. CALCULAR FLUJOS REALES (BOTTOM-UP)
    # Primero sumamos el consumo legítimo
    # Usamos un diccionario temporal para sumar
    flujo_acumulado = caudales.copy()
    
    # Truco iterativo para sumar hijos a padres (balance de masas legítimo)
    for _ in range(3):
        for hijo, padre in red_estructura.items():
            if padre:
                # Reiniciamos padres para recalcular suma de hijos
                pass 

    # Cálculo manual simple para asegurar precisión en este ejemplo de árbol
    # Flujo legítimo (sin contar la fuga aún)
    flujo_acumulado['Nodo_B'] = caudales['Casa_D'] + caudales['Casa_E']
    flujo_acumulado['Nodo_C'] = caudales['Casa_F'] + caudales['Casa_G']
    flujo_acumulado['Matriz_A'] = flujo_acumulado['Nodo_B'] + flujo_acumulado['Nodo_C']

    # 4. INYECTAR LA FUGA EN EL SISTEMA
    # La fuga aumenta el flujo que debe pasar por el nodo afectado y todos sus ancestros
    # Si la fuga está en Casa_D, entonces Casa_D, Nodo_B y Matriz_A ven aumentado su caudal real
    nodo_actual = nodo_con_fuga
    while nodo_actual is not None:
        flujo_acumulado[nodo_actual] += valor_fuga
        nodo_actual = red_estructura[nodo_actual] # Subir al padre

    # 5. GENERAR PRESIONES (Con caída brusca en la zona de fuga)
    presiones = {'Matriz_A': 6.0} # Presión inicial un poco más alta
    cola = ['Matriz_A']
    while cola:
        padre = cola.pop(0)
        hijos = [k for k, v in red_estructura.items() if v == padre]
        for hijo in hijos:
            # Caída normal
            perdida = random.uniform(0.1, 0.3)
            # Si este nodo es el de la fuga, la presión cae drásticamente
            if hijo == nodo_con_fuga:
                perdida += random.uniform(1.5, 2.5) # Gran caída de presión
            
            presiones[hijo] = round(max(0, presiones[padre] - perdida), 2)
            cola.append(hijo)

    return caudales, flujo_acumulado, presiones, nodo_con_fuga, valor_fuga

# --- FUNCIÓN 2: EL MODELO DETECTIVE ---
def detectar_fugas(caudales_base, flujos_reales, presiones):
    reporte = []
    
    # Para detectar, comparamos:
    # ¿Lo que sale de un nodo es igual a la suma de lo que entra en sus hijos?
    
    # Definir nodos padres (los que tienen ramificaciones)
    nodos_padres = ['Matriz_A', 'Nodo_B', 'Nodo_C']
    
    for padre in nodos_padres:
        # 1. Cuánta agua pasa por el sensor del padre
        caudal_padre = flujos_reales[padre]
        
        # 2. Cuánta agua pasa por los sensores de los hijos inmediatos
        hijos = [k for k, v in red_estructura.items() if v == padre]
        suma_hijos = sum(flujos_reales[h] for h in hijos)
        
        # 3. Balance
        diferencia = round(caudal_padre - suma_hijos, 2)
        
        # Si la diferencia es mayor a 0.1 (margen de error), hay fuga en este tramo
        if diferencia > 0.1:
            diagnostico = f"¡ALERTA! Fuga detectada entre {padre} y sus hijos."
            ubicacion_logica = "Tramo o Nodo Propio"
        else:
            diagnostico = "Tramo Normal"
            ubicacion_logica = "OK"

        reporte.append({
            'Nodo_Analizado': padre,
            'Lectura_Sensor': caudal_padre,
            'Suma_Lectura_Hijos': suma_hijos,
            'Desbalance_Agua': diferencia,
            'Diagnostico_Modelo': diagnostico
        })
        
    return pd.DataFrame(reporte)

In [8]:
# --- EJECUCIÓN ---
# 1. Generamos el desastre
consumos_teoricos, sensores_flujo, sensores_presion, donde_fue, cantidad = generar_red_con_fuga()

print(f"--- REALIDAD (Solo para verificar) ---")
print(f"La fuga real está en: {donde_fue}")
print(f"Cantidad de agua perdiéndose: {cantidad} L/min\n")

# 2. El modelo intenta adivinar
df_diagnostico = detectar_fugas(consumos_teoricos, sensores_flujo, sensores_presion)

print("--- REPORTE DEL MODELO DE DETECCIÓN ---")
print(df_diagnostico)

# 3. Exportar a Excel
archivo_fuga = 'reporte_fugas_etapa2.xlsx'
df_diagnostico.to_excel(archivo_fuga, index=False)
print(f"\nReporte exportado a: {archivo_fuga}")

--- REALIDAD (Solo para verificar) ---
La fuga real está en: Casa_E
Cantidad de agua perdiéndose: 65.67 L/min

--- REPORTE DEL MODELO DE DETECCIÓN ---
  Nodo_Analizado  Lectura_Sensor  Suma_Lectura_Hijos  Desbalance_Agua  \
0       Matriz_A          169.36              169.36              0.0   
1         Nodo_B          121.48              121.48              0.0   
2         Nodo_C           47.88               47.88              0.0   

  Diagnostico_Modelo  
0       Tramo Normal  
1       Tramo Normal  
2       Tramo Normal  

Reporte exportado a: reporte_fugas_etapa2.xlsx


### Red 2: 20 nodos

In [2]:
import pandas as pd
import random

In [3]:
# --- 1. CONFIGURACIÓN DE LA TOPOLOGÍA (RED GRANDE) ---
# Estructura: Matriz -> Sectores -> Pasajes -> Casas
red_estructura = {
    'Matriz_Principal': None,
    
    # Sector Norte
    'Valvula_Norte': 'Matriz_Principal',
    'Pasaje_Los_Alamos': 'Valvula_Norte',
    'Casa_N1': 'Pasaje_Los_Alamos',
    'Casa_N2': 'Pasaje_Los_Alamos',
    'Casa_N3': 'Pasaje_Los_Alamos',
    'Pasaje_Los_Robles': 'Valvula_Norte',
    'Casa_N4': 'Pasaje_Los_Robles',
    'Casa_N5': 'Pasaje_Los_Robles',
    
    # Sector Sur
    'Valvula_Sur': 'Matriz_Principal',
    'Pasaje_El_Sol': 'Valvula_Sur',
    'Casa_S1': 'Pasaje_El_Sol',
    'Casa_S2': 'Pasaje_El_Sol',
    'Casa_S3': 'Pasaje_El_Sol',
    'Pasaje_La_Luna': 'Valvula_Sur',
    'Casa_S4': 'Pasaje_La_Luna',
    'Casa_S5': 'Pasaje_La_Luna',
    'Casa_S6': 'Pasaje_La_Luna'
}

# Identificamos nodos finales automáticamente (los que no son padres de nadie)
padres = set(red_estructura.values())
nodos_finales = [n for n in red_estructura.keys() if n not in padres]

In [4]:
# --- 2. SIMULADOR DE FÍSICA HIDRÁULICA (NCh 691) ---
def simular_red(fuga_info=None):
    """
    Genera el estado de la red.
    fuga_info: Diccionario opcional {'nodo': str, 'magnitud_ls': float}
    Retorna: Diccionarios de (caudales, presiones)
    """
    # A. GENERAR CONSUMOS (L/s)
    # Consumo normal casa chilena punta: 0.15 - 0.30 L/s
    caudales = {n: 0.0 for n in red_estructura}
    for nodo in nodos_finales:
        caudales[nodo] = round(random.uniform(0.15, 0.40), 3)
    
    # Si hay fuga, añadimos ese "consumo extra" en el nodo específico
    if fuga_info:
        nodo_f = fuga_info['nodo']
        mag_f = fuga_info['magnitud_ls']
        # La fuga se suma al consumo natural del nodo
        caudales[nodo_f] += mag_f

    # B. CÁLCULO DE CAUDALES (Bottom-Up)
    # Sumamos desde las hojas hacia la raíz
    flujo_acumulado = caudales.copy()
    # Iteramos profunidad del árbol veces para asegurar la suma
    for _ in range(5): 
        for hijo, padre in red_estructura.items():
            if padre:
                # Reiniciamos el cálculo del padre para sumar desde cero en cada pasada
                # (Simplificación para script iterativo)
                hijos_de_este_padre = [k for k, v in red_estructura.items() if v == padre]
                suma_hijos = sum(flujo_acumulado[h] for h in hijos_de_este_padre)
                # Si el padre es un nodo de paso, su flujo es la suma de hijos + su propia fuga si la tuviera
                consumo_propio = caudales[padre] if padre not in nodos_finales else 0
                # Nota: si hay fuga en un nodo intermedio, se sumó en 'caudales' arriba
                if fuga_info and padre == fuga_info['nodo']:
                     consumo_propio = fuga_info['magnitud_ls']
                
                flujo_acumulado[padre] = round(suma_hijos + consumo_propio, 3)

    # C. CÁLCULO DE PRESIONES (m.c.a. - Metros Columna de Agua)
    # NCh 691: Presión estática máx 70 mca, dinámica min 15 mca.
    # Iniciamos con buena presión en la matriz (ej. 50 mca = 5 bar)
    presiones = {'Matriz_Principal': 50.0}
    
    cola = ['Matriz_Principal']
    while cola:
        padre = cola.pop(0)
        hijos = [k for k, v in red_estructura.items() if v == padre]
        
        for hijo in hijos:
            # Pérdida de carga (fricción) normal: 0.5 a 1.5 mca por tramo
            perdida = random.uniform(0.5, 1.5)
            
            # EFECTO FÍSICO DE LA FUGA:
            # Si hay una fuga AGUAS ABAJO o en ESTE NODO, la velocidad aumenta y la presión cae más.
            # Simplificamos: Si este nodo es el de la fuga, pierde mucha presión.
            if fuga_info and hijo == fuga_info['nodo']:
                perdida += random.uniform(10.0, 15.0) # Caída brusca por rotura
            
            presiones[hijo] = round(max(0, presiones[padre] - perdida), 2)
            cola.append(hijo)
            
    return flujo_acumulado, presiones

In [5]:
# --- 3. EJECUCIÓN Y DETECCIÓN ---

# PASO 1: Generar el "Gemelo Digital" (Estado Ideal sin fugas)
flujos_ideal, presiones_ideal = simular_red(fuga_info=None)

# PASO 2: Generar el Desastre (Estado Real con fuga)
# Elegimos una víctima al azar
nodo_victima = random.choice(list(red_estructura.keys()))
# Fuga magnitud NCh: Una rotura visible son unos 2 a 4 L/s
datos_fuga = {'nodo': nodo_victima, 'magnitud_ls': round(random.uniform(2.0, 4.0), 2)}

flujos_real, presiones_real = simular_red(fuga_info=datos_fuga)

print(f"--- SIMULACIÓN REALIZADA ---")
print(f"Fuga creada en: {nodo_victima} | Magnitud: {datos_fuga['magnitud_ls']} L/s")
print(f"(Nota: El sistema de detección no sabe esto, debe averiguarlo)\n")

# PASO 3: Algoritmo de Detección (Comparativa)
reporte = []

for nodo in red_estructura:
    # 1. Comparar Presión (La pista clave para ubicación exacta)
    p_esperada = presiones_ideal[nodo]
    p_medida = presiones_real[nodo]
    delta_presion = round(p_esperada - p_medida, 2)
    
    # 2. Comparar Flujo (Para confirmar anomalía)
    f_esperado = flujos_ideal[nodo]
    f_medido = flujos_real[nodo]
    delta_flujo = round(f_medido - f_esperado, 2) # Positivo significa que pasa más agua de la debida
    
    estado = "Normal"
    # Umbrales de detección
    if delta_presion > 8.0: # Si la presión cayó más de 8 metros respecto a lo normal
        estado = "POSIBLE FUGA AQUÍ"
    elif delta_flujo > 0.5 and delta_presion > 2.0:
        estado = "TRAMO AFECTADO (Aguas arriba de la fuga)"

    reporte.append({
        'Nodo': nodo,
        'Presion_Ideal_mca': p_esperada,
        'Presion_Sensor_mca': p_medida,
        'Caida_Presion': delta_presion, # La métrica clave
        'Flujo_Extra_Detectado_Ls': delta_flujo,
        'Diagnostico': estado
    })

--- SIMULACIÓN REALIZADA ---
Fuga creada en: Pasaje_Los_Alamos | Magnitud: 3.9 L/s
(Nota: El sistema de detección no sabe esto, debe averiguarlo)



In [6]:
# --- 4. EXPORTAR RESULTADOS ---
df = pd.DataFrame(reporte)
# Ordenamos por 'Caida_Presion' descendente. El top 1 debería ser la fuga.
df = df.sort_values(by='Caida_Presion', ascending=False)

print(df.head(10)) # Mostrar los 10 sospechosos principales
df.to_excel("Reporte_Fugas_Red_Grande.xlsx", index=False)
print("\nReporte exportado a Excel.")

                 Nodo  Presion_Ideal_mca  Presion_Sensor_mca  Caida_Presion  \
5             Casa_N3              47.82               37.74          10.08   
4             Casa_N2              47.51               37.51          10.00   
2   Pasaje_Los_Alamos              48.37               38.42           9.95   
3             Casa_N1              47.00               37.75           9.25   
12            Casa_S2              47.46               46.46           1.00   
11            Casa_S1              47.07               46.45           0.62   
10      Pasaje_El_Sol              48.13               47.68           0.45   
9         Valvula_Sur              49.12               48.75           0.37   
1       Valvula_Norte              49.34               49.16           0.18   
16            Casa_S5              47.56               47.44           0.12   

    Flujo_Extra_Detectado_Ls        Diagnostico  
5                       0.04  POSIBLE FUGA AQUÍ  
4                       0.08  